In [0]:
silver_path = "abfss://silver@travelappprojectstorage.dfs.core.windows.net/trip_adherence/trip_adherence_clean"
gold_path = "abfss://gold@travelappprojectstorage.dfs.core.windows.net/"

In [0]:
from pyspark.sql.functions import *

In [0]:
gold_layer = spark.read.format("delta").load(silver_path)

In [0]:
gold_trip_progress = (
    gold_layer.groupBy("TripId", "TouristId", "Name", "AgencyId", "trip_status")
    .agg(
        count("*").alias("total_scheduled_places"),
        sum(when(col("movement_status") == "ARRIVED", 1).otherwise(0)).alias("completed_places"),
        sum(when(col("delay_category") != "ON_TIME", 1).otherwise(0)).alias("delayed_places"),
        avg("distance_km").alias("avg_distance_km"),
        max("UpdatedAt").alias("last_location_update")
    )
    .withColumn(
        "trip_completion_percent",
        round((col("completed_places") / col("total_scheduled_places")) * 100, 2)
    )
)
(
    gold_trip_progress.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .save(f"{gold_path}/gold_trip_progress")
)

In [0]:
gold_schedule_adherence = (
    gold_layer.groupBy("TripId")
    .agg(
        count("*").alias("total_schedule_points"),
        sum(when(col("delay_category") == "ON_TIME", 1).otherwise(0)).alias("on_time_count"),
        sum(when(col("delay_category") == "MINOR_DELAY", 1).otherwise(0)).alias("minor_delay_count"),
        sum(when(col("delay_category") == "MAJOR_DELAY", 1).otherwise(0)).alias("major_delay_count"),
        avg("delay_minutes").alias("avg_delay_minutes")
    )
    .withColumn(
        "schedule_adherence_percent",
        round((col("on_time_count") / col("total_schedule_points")) * 100, 2)
    )
)
(
    gold_schedule_adherence.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .save(f"{gold_path}/gold_schedule_adherence")
)

In [0]:
gold_trip_summary = (
    gold_layer.groupBy("TripId", "TouristId", "Name", "AgencyId", "trip_status")
    .agg(
        min("StartDate").alias("trip_start_date"),
        max("EndDate").alias("trip_end_date"),
        max("trip_duration_days").alias("trip_duration_days"),
        countDistinct("PlaceId").alias("planned_places"),
        sum(when(col("movement_status") == "ARRIVED", 1).otherwise(0)).alias("visited_places"),
        avg("delay_minutes").alias("avg_delay_minutes")
    )
    .withColumn(
        "trip_efficiency_percent",
        round((col("visited_places") / col("planned_places")) * 100, 2)
    )
)
(
    gold_trip_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .save(f"{gold_path}/gold_trip_summary")
)

In [0]:
gold_agency_trip_performance = (
    gold_layer.groupBy("AgencyId", "AgencyName")
    .agg(
        countDistinct("TripId").alias("total_trips"),
        countDistinct("TouristId").alias("total_tourists"),
        avg("delay_minutes").alias("avg_delay_minutes"),
        sum(when(col("delay_category") == "MAJOR_DELAY", 1).otherwise(0)).alias("major_delay_count")
    )
)
(
    gold_agency_trip_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("mergeSchema", "true")
    .save(f"{gold_path}/gold_agency_trip_performance")
)

In [0]:
gold_trip_progress_df = spark.read.format("delta").load(
    f"{gold_path}/gold_trip_progress"
)

gold_schedule_adherence_df = spark.read.format("delta").load(
    f"{gold_path}/gold_schedule_adherence"
)

gold_trip_summary_df = spark.read.format("delta").load(
    f"{gold_path}/gold_trip_summary"
)

gold_agency_trip_performance_df = spark.read.format("delta").load(
    f"{gold_path}/gold_agency_trip_performance"
)

display(gold_trip_progress_df)
print("gold_trip_progress count:", gold_trip_progress_df.count())

display(gold_schedule_adherence_df)
print("gold_schedule_adherence count:", gold_schedule_adherence_df.count())

display(gold_trip_summary_df)
print("gold_trip_summary count:", gold_trip_summary_df.count())

display(gold_agency_trip_performance_df)
print("gold_agency_trip_performance count:", gold_agency_trip_performance_df.count())